# Study 2: explore five baseline policies

Each section defines one rule for the four regions, then runs that rule on the same five environment seeds. These are fixed decision rules, so there is no Q-table or training step here.

Action IDs: `0` save, `1` develop, `2` low mitigation, `3` medium mitigation, `4` high mitigation.

## Shared settings and episode loop

In [1]:
import random
import pandas as pd
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment


ENVIRONMENT_SEEDS = (1000, 1001, 1002, 1003, 1004)

In [2]:
def run_baseline_episode(name, policy, seed):
    environment = EconomicSustainabilityEnvironment()
    observations = environment.reset(seed)
    policy_rng = random.Random(seed + 10_000)

    gpi_total = 0.0
    mitigation_spending = 0.0
    invalid_attempts = 0

    while True:
        actions = policy(environment, observations, policy_rng)
        observations, rewards, terminated, truncated, info = environment.step(actions)

        gpi_total += info["global_pollution_index"]
        invalid_attempts += sum(info["invalid_attempts"])

        for region in range(4):
            if actions[region] in (2, 3, 4):
                mitigation_spending += info["costs_paid"][region]

        if terminated or truncated:
            break

    return {
        "policy": name,
        "seed": seed,
        "steps": environment.step_count,
        "failure": terminated,
        "mean_gpi": gpi_total / environment.step_count,
        "final_mean_edi": sum(environment.development_indices) / 4,
        "mitigation_spending": mitigation_spending,
        "invalid_attempts": invalid_attempts,
    }


results = []

## 1. Save

Every region chooses action 0. This shows what happens when no region spends currency on development or mitigation.

In [3]:
def save_policy(environment, observations, rng):
    return (0, 0, 0, 0)


save_rows = []
for seed in ENVIRONMENT_SEEDS:
    save_rows.append(run_baseline_episode("save", save_policy, seed))
results.extend(save_rows)
display(pd.DataFrame(save_rows))

,policy,seed,steps,failure,mean_gpi,final_mean_edi,mitigation_spending,invalid_attempts
0,save,1000,200,False,55.08,0.0,0.0,0
1,save,1001,200,False,55.08,0.0,0.0,0
2,save,1002,200,False,55.08,0.0,0.0,0
3,save,1003,200,False,55.08,0.0,0.0,0
4,save,1004,200,False,55.08,0.0,0.0,0


## 2. Develop

Each region buys development when it has enough currency and its development index is below 100. Otherwise it saves.

In [4]:
def develop_policy(environment, observations, rng):
    actions = []
    development_cost = environment.config.action_costs[1]

    for region in range(4):
        balance = environment.currency_balances[region]
        edi = environment.development_indices[region]

        if edi < 100 and balance >= development_cost:
            actions.append(1)
        else:
            actions.append(0)

    return tuple(actions)


develop_rows = []
for seed in ENVIRONMENT_SEEDS:
    develop_rows.append(run_baseline_episode("develop", develop_policy, seed))
results.extend(develop_rows)
display(pd.DataFrame(develop_rows))

,policy,seed,steps,failure,mean_gpi,final_mean_edi,mitigation_spending,invalid_attempts
0,develop,1000,23,True,73.898338,85.50,0.0,0
1,develop,1001,22,True,70.543138,82.25,0.0,0
2,develop,1002,26,True,75.265048,84.00,0.0,0
3,develop,1003,26,True,74.881087,83.50,0.0,0
4,develop,1004,23,True,71.539381,81.00,0.0,0


## 3. Mitigate

Each region chooses the strongest mitigation it can afford. If it cannot afford any mitigation, it saves.

In [5]:
def mitigate_policy(environment, observations, rng):
    actions = []
    costs = environment.config.action_costs

    for region in range(4):
        balance = environment.currency_balances[region]
        action = 0

        for option in (4, 3, 2):
            if balance >= costs[option]:
                action = option
                break

        actions.append(action)

    return tuple(actions)


mitigate_rows = []
for seed in ENVIRONMENT_SEEDS:
    mitigate_rows.append(run_baseline_episode("mitigate", mitigate_policy, seed))
results.extend(mitigate_rows)
display(pd.DataFrame(mitigate_rows))

,policy,seed,steps,failure,mean_gpi,final_mean_edi,mitigation_spending,invalid_attempts
0,mitigate,1000,200,False,1.29165,0.0,18980.0,0
1,mitigate,1001,200,False,1.29510,0.0,18715.0,0
2,mitigate,1002,200,False,1.37550,0.0,18095.0,0
3,mitigate,1003,200,False,1.34655,0.0,17955.0,0
4,mitigate,1004,200,False,1.41030,0.0,18195.0,0


## 4. Coordinate

Each region mitigates when its local pollution bin or the shared GPI bin rises above zero. Otherwise, it develops until EDI reaches 70, then saves. The rule checks currency before buying an action.

In [6]:
def coordinate_policy(environment, observations, rng):
    actions = []
    costs = environment.config.action_costs

    for region in range(4):
        balance = environment.currency_balances[region]
        edi = environment.development_indices[region]
        local_pollution_bin = observations[region][2]
        global_pollution_bin = observations[region][3]

        if local_pollution_bin >= 1 or global_pollution_bin >= 1:
            action = 0
            for option in (4, 3, 2):
                if balance >= costs[option]:
                    action = option
                    break
        elif edi < 70 and balance >= costs[1]:
            action = 1
        else:
            action = 0

        actions.append(action)

    return tuple(actions)


coordinate_rows = []
for seed in ENVIRONMENT_SEEDS:
    coordinate_rows.append(run_baseline_episode("coordinate", coordinate_policy, seed))
results.extend(coordinate_rows)
display(pd.DataFrame(coordinate_rows))

,policy,seed,steps,failure,mean_gpi,final_mean_edi,mitigation_spending,invalid_attempts
0,coordinate,1000,200,False,28.736276,70.0,14600.0,0
1,coordinate,1001,200,False,28.289254,70.0,14700.0,0
2,coordinate,1002,200,False,28.498437,70.0,14675.0,0
3,coordinate,1003,200,False,28.484010,70.0,14650.0,0
4,coordinate,1004,200,False,28.253857,70.0,14650.0,0


## 5. Random affordable

Each region draws from the actions it can currently afford. Development is excluded once its EDI reaches 100. A separate seeded generator makes the action draws repeatable.

In [7]:
def random_affordable_policy(environment, observations, rng):
    actions = []
    costs = environment.config.action_costs

    for region in range(4):
        balance = environment.currency_balances[region]
        edi = environment.development_indices[region]
        affordable = []

        for option in range(5):
            if balance >= costs[option] and (option != 1 or edi < 100):
                affordable.append(option)

        actions.append(rng.choice(affordable))

    return tuple(actions)


random_rows = []
for seed in ENVIRONMENT_SEEDS:
    random_rows.append(run_baseline_episode("random_affordable", random_affordable_policy, seed))
results.extend(random_rows)
display(pd.DataFrame(random_rows))

,policy,seed,steps,failure,mean_gpi,final_mean_edi,mitigation_spending,invalid_attempts
0,random_affordable,1000,200,False,28.415873,100.0,16500.0,0
1,random_affordable,1001,200,False,31.748382,100.0,15335.0,0
2,random_affordable,1002,200,False,34.131224,100.0,14335.0,0
3,random_affordable,1003,200,False,29.191007,100.0,16110.0,0
4,random_affordable,1004,200,False,31.629938,100.0,15290.0,0


## Compare the five rules

These are descriptive outcomes on a few seeds. A low pollution value can come with high spending or little development, so inspect the columns together.

In [8]:
episodes = pd.DataFrame(results)
summary = episodes.groupby("policy", sort=False).agg(
    mean_gpi=("mean_gpi", "mean"),
    final_mean_edi=("final_mean_edi", "mean"),
    mitigation_spending=("mitigation_spending", "mean"),
    failure_rate=("failure", "mean"),
    invalid_attempts=("invalid_attempts", "sum"),
)

display(summary.round(2))

,mean_gpi,final_mean_edi,mitigation_spending,failure_rate,invalid_attempts
policy,,,,,
save,55.08,0.00,0.0,0.0,0
develop,73.23,83.25,0.0,1.0,0
mitigate,1.34,0.00,18388.0,0.0,0
coordinate,28.45,70.00,14655.0,0.0,0
random_affordable,31.02,100.00,15514.0,0.0,0


## What these five rules show

Across these five starts, **develop** increased EDI but reached environmental failure in every episode, usually after 22?26 steps. Its mean GPI of 73.23 covers shorter episodes, so it is not a full 200-step comparison with the other rules.

**Save** spent nothing and kept mean GPI at 55.08, but produced no development. **Mitigate** had the lowest mean GPI (1.34), with no development and the highest mitigation spending (18,388 per episode). **Coordinate** balanced some development (final EDI 70) with mean GPI 28.45 and spending 14,655. **Random affordable** reached EDI 100 with mean GPI 31.02 and spending 15,514, although its episode results varied more across seeds.

None of the rules attempted an invalid action. These five starts are enough to understand how the rules behave, but they are too few for a firm policy ranking.